# Fase 4: Modelado

**Pregunta experimental:** ¿qué relación tienen las características registradas de los pozos horizontales shale de la cuenca Neuquina con el volumen de agua inyectada, y qué capacidad predictiva muestran en operaciones posteriores? Esta fase usa la salida limpia de la fase 3 y se ejecuta desde la raíz del repositorio. La evaluación detallada corresponde a la fase 5.

| Elemento | Decisión |
|---|---|
| Alcance | Pozos horizontales shale de la cuenca Neuquina, seleccionados y preparados en la fase 3. |
| Objetivo | `agua_inyectada_m3`, volumen en m³; regresión supervisada, según la fase 2. |
| Entradas | Longitud de rama horizontal, cantidad de fracturas y tipo de terminación. La fase 3 las selecciona para el análisis histórico; el CSV no confirma que sus valores estuvieran disponibles antes de cada operación. |
| Algoritmo | `RandomForestRegressor`: aprende relaciones no lineales e interacciones sin exigir una fórmula previa. Su limitación es que no extrapola bien fuera de lo observado. |
| Referencia | `DummyRegressor` con mediana del entrenamiento. |
| Prueba | Operaciones iniciadas desde el 1/1/2025, reservadas sin usarlas para ajustar el modelo. |

Las columnas `profundidad_m`, `presion_inicial_psi` y `temperatura_c` no existen en el CSV original, según la comprobación documentada en la fase 3. No se usan `cuenca`, `tipo_reservorio` ni `subtipo_reservorio` porque son constantes tras el recorte. Tampoco se usa `formacion_productiva`: 2.630 de los 2.632 casos finales son Vaca Muerta, por lo que las otras dos categorías tienen respaldo insuficiente para aprender una diferencia fiable. No se usan identificadores, fechas de fin, duración, presión máxima ni cantidades efectivamente bombeadas como predictores: revelarían información de la operación terminada o facilitarían memorizar pozos. La comprobación temporal de la fase 3 no permite afirmar que `cantidad_fracturas` y `tipo_terminacion` fueran conocidas antes de la operación. El resultado se interpreta como evaluación retrospectiva sobre registros históricos. Para usarlo en planificación se necesitan valores planificados equivalentes; si no existen, habría que retirar esas variables y volver a entrenar.


## 4.1. Carga de los datos preparados

La fase 4 lee `df_limpio.csv`, generado por la fase 3 desde el CSV original versionado. El archivo ya contiene el recorte a pozos horizontales shale de la cuenca Neuquina y la exclusión de objetivos inválidos y fechas futuras al 28/10/2026. Se comprueban estas condiciones para detectar una salida vieja o equivocada. La selección de la población y la limpieza se documentan en la fase 3; aquí solo se preparan las columnas de entrada para el algoritmo.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

from comun import (
	CORTE_TEMPORAL,
	ENTRADAS_CATEGORICAS,
	ENTRADAS_NUMERICAS,
	FECHA_REVISION,
	OBJETIVO,
	RUTA_ARTEFACTOS,
	RUTA_DATOS_LIMPIOS,
	exportar_modelos,
)

if not RUTA_DATOS_LIMPIOS.is_file():
	raise FileNotFoundError("Ejecutá primero la fase 3 desde la raíz del repositorio.")
df = pd.read_csv(RUTA_DATOS_LIMPIOS)
df["fecha_inicio_fractura"] = pd.to_datetime(df["fecha_inicio_fractura"], errors="coerce")
print(f"Registros recibidos de fase 3: {len(df):,}")

Registros recibidos de fase 3: 2,632


In [2]:
# La lista de entradas y el objetivo viven en comun.py para que la fase 5
# evalúe exactamente las mismas variables con las que se entrenó el modelo.
objetivo = OBJETIVO
entradas_numericas = ENTRADAS_NUMERICAS
entradas_categoricas = ENTRADAS_CATEGORICAS
entradas = entradas_numericas + entradas_categoricas
columnas_necesarias = [*entradas, objetivo, "cuenca", "tipo_reservorio", "subtipo_reservorio", "idpozo", "id_base_fractura_adjiv", "fecha_inicio_fractura"]
faltantes = set(columnas_necesarias) - set(df.columns)
if faltantes:
	raise ValueError(f"Columnas faltantes en la salida de fase 3: {sorted(faltantes)}")

alcance_correcto = (
	df["cuenca"].eq("NEUQUINA") & df["tipo_reservorio"].eq("NO CONVENCIONAL") & df["subtipo_reservorio"].eq("SHALE") & df["longitud_rama_horizontal_m"].gt(0)
)
agua = pd.to_numeric(df[objetivo], errors="coerce")
if not alcance_correcto.all() or not (agua.gt(0) & np.isfinite(agua)).all() or not df["fecha_inicio_fractura"].le(FECHA_REVISION).all():
	raise ValueError("La salida de fase 3 no cumple el alcance o los criterios de calidad documentados; ejecutá de nuevo esa fase.")

datos_modelo = df[columnas_necesarias].copy()
datos_modelo[objetivo] = agua
for columna in entradas_numericas:
	datos_modelo[columna] = pd.to_numeric(datos_modelo[columna], errors="coerce").replace([np.inf, -np.inf], np.nan)
print(f"Registros listos para el modelo: {len(datos_modelo):,}")
print(datos_modelo[[*entradas, objetivo]].isna().sum().to_frame("faltantes antes del pipeline"))

Registros listos para el modelo: 2,632
                            faltantes antes del pipeline
longitud_rama_horizontal_m                             0
cantidad_fracturas                                     0
tipo_terminacion                                       0
agua_inyectada_m3                                      0


## 4.2. Separación temporal

El corte de 2025 deja las operaciones posteriores exclusivamente para prueba. Si un pozo figura en ambos períodos, se retira del entrenamiento para evitar que el modelo aprenda información del mismo pozo que luego se evalúa. El conjunto de prueba no participa de la imputación, codificación ni elección de parámetros.


In [3]:
corte_temporal = CORTE_TEMPORAL
train = datos_modelo.loc[datos_modelo["fecha_inicio_fractura"] < corte_temporal].copy()
test = datos_modelo.loc[datos_modelo["fecha_inicio_fractura"] >= corte_temporal].copy()
pozos_compartidos = set(train["idpozo"]).intersection(test["idpozo"])
train = train.loc[~train["idpozo"].isin(pozos_compartidos)].copy()
if train.empty or test.empty:
	raise ValueError("La partición temporal quedó vacía; revisar fechas y filtros.")
assert train["fecha_inicio_fractura"].max() < test["fecha_inicio_fractura"].min()
assert not set(train["idpozo"]).intersection(test["idpozo"])
X_train, y_train = train[entradas], train[objetivo]
X_test, y_test = test[entradas], test[objetivo]
print(f"Entrenamiento: {len(train):,} operaciones; prueba: {len(test):,}.")
print(f"Pozos compartidos retirados del entrenamiento: {len(pozos_compartidos)}.")
print(f"Rango de entrenamiento: {train.fecha_inicio_fractura.min().date()} a {train.fecha_inicio_fractura.max().date()}")
print(f"Rango de prueba: {test.fecha_inicio_fractura.min().date()} a {test.fecha_inicio_fractura.max().date()}")

Entrenamiento: 1,937 operaciones; prueba: 691.
Pozos compartidos retirados del entrenamiento: 4.
Rango de entrenamiento: 2011-12-13 a 2024-12-21
Rango de prueba: 2025-01-03 a 2026-10-05


## 4.3. Entrenamiento

Las medianas numéricas y las categorías más frecuentes se calculan solo sobre entrenamiento. `OneHotEncoder` transforma las categorías y tolera categorías nuevas en prueba. Se fijan semilla y parámetros para obtener resultados reproducibles. `min_samples_leaf=3` limita hojas demasiado específicas; 200 árboles reducen la variación entre árboles con un costo razonable para este conjunto. Son parámetros iniciales, no se afirma que sean óptimos: su ajuste requeriría una validación temporal adicional dentro del período de entrenamiento.


In [4]:
try:
	codificador = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # Compatibilidad con scikit-learn anterior a 1.2
	codificador = OneHotEncoder(handle_unknown="ignore", sparse=False)

prep = ColumnTransformer(
	[
		("num", SimpleImputer(strategy="median"), entradas_numericas),
		(
			"cat",
			Pipeline(
				[
					("imputer", SimpleImputer(strategy="most_frequent")),
					("onehot", codificador),
				]
			),
			entradas_categoricas,
		),
	]
)
modelo = Pipeline(
	[
		("prep", prep),
		("regresor", RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=-1)),
	]
)
referencia = DummyRegressor(strategy="median")
modelo.fit(X_train, y_train)
referencia.fit(X_train, y_train)
print("Modelos entrenados. La prueba temporal permanece reservada para la fase 5.")


Modelos entrenados. La prueba temporal permanece reservada para la fase 5.


## 4.4. Comparación de modelos

La elección del algoritmo se resuelve con validación cruzada **sobre el conjunto de entrenamiento** (5 cortes cronológicos, ordenados por `fecha_inicio_fractura`), y no sobre la prueba temporal: usar la prueba para elegir el modelo haría que la evaluación final de la fase 5 quedara inflada. Se comparan:

- **Ridge** (regresión lineal regularizada): referencia lineal interpretable y el único que puede extrapolar fuera del rango histórico; necesita escalado.
- **SVR con núcleo RBF**: captura no linealidades; se escala y se entrena sobre `log1p(agua)` por la asimetría del objetivo.
- **Random Forest**: el modelo de la sección 4.3.
- **Gradient Boosting** (implementación por histogramas de scikit-learn): suele ser el más fuerte en datos tabulares.

Todos comparten el mismo preprocesamiento (imputación + one-hot); Ridge y SVR agregan escalado numérico dentro del pipeline, para que no haya fuga de información desde la validación. El ganador se reentrena con todo el entrenamiento y es el que se exporta a la fase 5. La métrica de selección es el **MAE en m³**, la principal según la fase 5, por su lectura directa en las unidades del problema.

In [5]:
# @title Comparación de modelos con validación cruzada temporal
def construir(regresor, escalar=False, log_objetivo=False):
	"""Arma el mismo preprocesamiento para todos, con extras opcionales."""
	pasos_num = [("imputer", SimpleImputer(strategy="median"))]
	if escalar:
		pasos_num.append(("scaler", StandardScaler()))

	pipeline = Pipeline(
		[
			(
				"prep",
				ColumnTransformer(
					[
						("num", Pipeline(pasos_num), entradas_numericas),
						(
							"cat",
							Pipeline(
								[
									("imputer", SimpleImputer(strategy="most_frequent")),
									("onehot", codificador),
								]
							),
							entradas_categoricas,
						),
					]
				),
			),
			("regresor", regresor),
		]
	)
	if log_objetivo:
		return TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1)
	return pipeline


candidatos = {
	"Ridge": construir(Ridge(alpha=1.0), escalar=True),
	"SVR (RBF)": construir(SVR(C=100.0, gamma="scale", epsilon=0.05), escalar=True, log_objetivo=True),
	"Random Forest": construir(RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=-1)),
	"Gradient Boosting": construir(HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, min_samples_leaf=20, random_state=42)),
}

# La validación cruzada es cronológica: cada corte entrena con lo anterior y
# valida con lo posterior, igual que el uso real del modelo.
orden = train.sort_values("fecha_inicio_fractura").index
X_cv, y_cv = X_train.loc[orden], y_train.loc[orden]
cv = TimeSeriesSplit(n_splits=5)

filas = []
for nombre, candidato in candidatos.items():
	mae = -cross_val_score(candidato, X_cv, y_cv, cv=cv, scoring="neg_mean_absolute_error")
	filas.append({"modelo": nombre, "MAE_cv_m3": mae.mean(), "desvio_cv_m3": mae.std()})

comparativa = pd.DataFrame(filas).sort_values("MAE_cv_m3").reset_index(drop=True)
display(comparativa.round(1))

# El ganador se reentrena con todo el entrenamiento y es el que pasa a fase 5.
nombre_modelo = comparativa.loc[0, "modelo"]
modelo = candidatos[nombre_modelo]
modelo.fit(X_train, y_train)
print(f"Mejor modelo según la validación cruzada: {nombre_modelo}")

,modelo,MAE_cv_m3,desvio_cv_m3
0,Ridge,10621.6,1951.8
1,Random Forest,13961.3,2091.2
2,Gradient Boosting,20696.2,10299.0
3,SVR (RBF),22457.0,17908.6


Mejor modelo según la validación cruzada: Ridge


## 4.5. Salida para la fase 5

Se guardan el pipeline completo ganador de la comparación, su nombre, la tabla comparativa, la referencia, la partición de prueba y los identificadores de cada operación. La fase 5 consume exactamente estos objetos: así evalúa el mismo modelo y las mismas filas con las que se entrenó.

In [6]:
exportar_modelos(
	modelo=modelo,
	referencia=referencia,
	nombre_modelo=nombre_modelo,
	comparativa_modelos=comparativa,
	X_test=X_test,
	y_test=y_test,
	test_metadata=test[["id_base_fractura_adjiv", "idpozo", "fecha_inicio_fractura"]],
)
print(f"Mejor modelo exportado: {nombre_modelo}")
print(f"Artefactos guardados en {RUTA_ARTEFACTOS}")

Mejor modelo exportado: Ridge
Artefactos guardados en C:\Users\Usuario\tp-ingenieria\Ingenieria_De_Software\artifacts
